In [ ]:
from langchain.chains import LLMChain
import os

In [ ]:
from langchain.prompts import PromptTemplate

In [3]:
from dotenv import load_dotenv
load_dotenv("config.env")  # take environment variables from .env.

True

In [4]:
# !pip install --upgrade langchain-google-genai
# !pip install llmonitor

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", api_key=os.getenv("GEMINI_API_KEY"))

In [ ]:
from langchain.callbacks.base import BaseCallbackHandler
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

# simple custom callback that logs start, tokens and end
class PrintCallbackHandler(BaseCallbackHandler):
    def on_llm_start(self, serialized, prompts, **kwargs):
        print("Do something on LLM start...")

    # def on_llm_new_token(self, token: str, **kwargs):
    #     print(f"New token: {token}")

    def on_llm_end(self, response, **kwargs):
        print("LLM end. Do something...")

# create a prompt and chain (uses PromptTemplate, LLMChain, and llm already present in the notebook)
prompt = PromptTemplate.from_template("Write a one-line joke about {topic}.")
chain = LLMChain(llm=llm, prompt=prompt)

# run the chain with callbacks; StreamingStdOutCallbackHandler streams to stdout,
# while PrintCallbackHandler demonstrates custom callback hooks.
callbacks = [PrintCallbackHandler(), StreamingStdOutCallbackHandler()]

result = chain.invoke({"topic": "programming"}, config={"callbacks": callbacks})

print("\nFinal result:", result)

Do something on LLM start...
LLM end. Do something...

Final result: {'topic': 'programming', 'text': 'Why do programmers prefer dark mode? Because light attracts bugs.'}


Here is a list of all callbacks

https://python.langchain.com/docs/concepts/callbacks/#callback-events

## Memory

In [ ]:
from langchain.chains import LLMChain
from langchain.memory import ConversationBufferMemory, ConversationBufferWindowMemory
from langchain_core.messages import SystemMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts.chat import (
    ChatPromptTemplate,
    HumanMessagePromptTemplate,
    MessagesPlaceholder,
)

prompt = ChatPromptTemplate(
    [
        MessagesPlaceholder(variable_name="chat_history"),
        HumanMessagePromptTemplate.from_template("{text}"),
    ]
)

memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
buffer_memory = ConversationBufferWindowMemory(memory_key="chat_history", k=2, return_messages=True)

chain = LLMChain(
    llm=llm,
    prompt=prompt,
    memory=buffer_memory,
)

result = chain.invoke({"text": "my name is bob"})
print(result)

{'text': "Hi Bob, it's nice to meet you! How can I help you today?", 'chat_history': []}


In [ ]:
result = chain.invoke({"text": "what was my name"})
result

{'text': 'Your name is Bob.',
 'chat_history': [HumanMessage(content='my name is bob', additional_kwargs={}, response_metadata={}),
  AIMessage(content="Hi Bob, it's nice to meet you! How can I help you today?", additional_kwargs={}, response_metadata={})]}

### Function Calling

In [5]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", api_key=os.getenv("GEMINI_API_KEY"))

In [9]:
56473 * 980123

55350486179

In [8]:
print(llm.invoke("What is 56473 x 980123").content)

To calculate $56473 \times 980123$, we can perform the multiplication directly.

We can break down the multiplication into smaller steps.

First, multiply $56473$ by $3$:
$56473 \times 3 = 169419$

Next, multiply $56473$ by $20$:
$56473 \times 20 = 1129460$

Next, multiply $56473$ by $100$:
$56473 \times 100 = 5647300$

Next, multiply $56473$ by $0000$ (which is 0):
$56473 \times 0000 = 0$

Next, multiply $56473$ by $800000$:
$56473 \times 800000 = 45178400000$

Next, multiply $56473$ by $90000000$:
$56473 \times 90000000 = 5082570000000$

Now, add all these results together:
```
         169419
       1129460
      5647300
           0
  45178400000
+ 5082570000000
-----------------
  5082570000000
    45178400000
      5647300
      1129460
        169419
-----------------
  5087754076179
```

Alternatively, we can use a calculator or perform the multiplication using a standard algorithm:
```
   980123
x   56473
---------
   2940369  (980123 x 3)
  6860861   (980123 x 70)
 3920492   

In [ ]:
55350486179

In [10]:
from langchain_core.tools import tool


@tool
def add(a: int, b: int) -> int:
    """Adds a and b."""
    return a + b


@tool
def multiply(a: int, b: int) -> int:
    """Multiplies a and b."""
    return a * b


tools = [add, multiply]

In [11]:
llm_with_tools = llm.bind_tools(tools)

In [18]:
llm.invoke("2x3")

AIMessage(content='The product of 2 and 3 is 6.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': []}, id='run--3f476926-9cd0-44cd-8741-a18375cfa2ac-0', usage_metadata={'input_tokens': 4, 'output_tokens': 12, 'total_tokens': 16, 'input_token_details': {'cache_read': 0}})

In [41]:
ai_msg = llm_with_tools.invoke("What is 56473 x (980123 + 1232112)")

In [42]:
ai_msg

AIMessage(content='', additional_kwargs={'function_call': {'name': 'add', 'arguments': '{"a": 980123, "b": 1232112}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': []}, id='run--17e2e8ba-47ea-4a8f-ae4a-c613cbd8e1c2-0', tool_calls=[{'name': 'add', 'args': {'a': 980123, 'b': 1232112}, 'id': 'a85c27fe-8c08-4492-8461-f7074da3784e', 'type': 'tool_call'}], usage_metadata={'input_tokens': 121, 'output_tokens': 29, 'total_tokens': 150, 'input_token_details': {'cache_read': 0}})

In [28]:
ai_msg.tool_calls[0]

{'name': 'multiply',
 'args': {'a': 56473, 'b': 980123},
 'id': '0c9ed69d-2951-48f6-af7e-0a539350e760',
 'type': 'tool_call'}

In [29]:
for tool_call in ai_msg.tool_calls:
    selected_tool = {"add": add, "multiply": multiply}[tool_call["name"].lower()]
    tool_output = selected_tool.invoke(tool_call["args"])
    print(tool_output)

55350486179


### LCEL | Langchain Expressions Language

In [ ]:
## input | llm | output_parser

In [43]:
llm.invoke("Hi")

AIMessage(content='Hello there! How can I help you today?', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': []}, id='run--baebfc92-b1dc-4497-93d2-5b182e73ca17-0', usage_metadata={'input_tokens': 2, 'output_tokens': 10, 'total_tokens': 12, 'input_token_details': {'cache_read': 0}})

In [44]:
from langchain.prompts import ChatPromptTemplate
from langchain.schema import StrOutputParser

prompt = ChatPromptTemplate.from_template("Tell me a joke about {topic}")
output_parser = StrOutputParser()

# Chain components with pipe operator
chain = prompt | llm | output_parser
response = chain.invoke({"topic": "programming"})

In [45]:
response

'Why do programmers prefer dark mode?\n\nBecause light attracts bugs!'

In [46]:
from langchain_core.runnables import RunnableParallel

prompt1 = ChatPromptTemplate.from_template("Summarize the following text: {text}")
prompt2 = ChatPromptTemplate.from_template("Translate the following text to French: {text}")

# Run multiple chains with same input
runnable1 = prompt1 | llm | output_parser
runnable2 = prompt2 | llm | output_parser

chain = RunnableParallel(first=runnable1, second=runnable2)

chain.invoke({"text": "I love programming."})

{'first': 'The provided text is a single sentence: "I love programming."\n\nTherefore, the summary is simply: **The author enjoys programming.**',
 'second': 'Here are a few ways to translate "I love programming" into French, with slight nuances:\n\n* **J\'adore programmer.** (This is the most direct and common translation, expressing strong liking.)\n* **J\'aime programmer.** (This is also very common and means "I like programming." It\'s slightly less intense than "j\'adore.")\n* **La programmation me plaît beaucoup.** (This translates to "Programming pleases me a lot." It\'s a bit more formal and emphasizes the enjoyment derived from it.)\n* **Je suis passionné(e) par la programmation.** (This means "I am passionate about programming." Use "passionné" if you are male, and "passionnée" if you are female.)\n\n**Which one to choose?**\n\n* **"J\'adore programmer"** is usually the best and most natural choice for expressing strong enthusiasm.\n* **"J\'aime programmer"** is also perfectl

## Exercises

In [ ]:
from typing import Optional

# Exercise (Health & Fitness domain, ~7 min)
# Objective:
# Build a "Fitness Motivation Assistant" that:
#  1. Reads a user's name from conversation memory to personalize the message.
#  2. Calculates total calories burned for a workout using the "multiply" tool.
#  3. Generates a short, encouraging message using the existing "llm".
#  4. Translates the motivational message into French using the "runnable2" chain.
#
# Constraints / hints:
#  - Use "buffer_memory" to find a recent Human message like "my name is bob".
#  - The "multiply" tool is available as "multiply.func(a: int, b: int)".
#  - Build a PromptTemplate for the motivational message and construct an LLMChain with the existing "llm".
#  - Use "runnable2.invoke({'text': text})" to get the French translation.
#  - Use the "callbacks" variable when invoking chains to see the callback system in action.
#
# Tasks (implement the three functions below). Do not change global objects (llm, multiply, buffer_memory, runnable2, callbacks).


def get_user_name_from_memory(mem=buffer_memory) -> Optional[str]:
    """
    Inspect mem.chat_memory.messages to find the user's name.
    Look for patterns like 'my name is <name>' or 'I am <name>'.
    Return the found name (capitalized) or None.
    Hints:
      - Messages are in a list: mem.chat_memory.messages
      - Each message has a .content attribute (string).
      - A simple string split should be sufficient for parsing.
    """
    raise NotImplementedError  # TODO: implement simple parsing of memory messages


def calculate_calories_burned(calories_per_minute: int, duration_minutes: int) -> int:
    """
    Calculate total calories burned using the `multiply` tool.
    Return the integer result.
    Hint:
      - Use multiply.func(a, b)
    """
    raise NotImplementedError  # TODO: implement using multiply.func


def generate_motivational_message(user_name: Optional[str], activity: str, calories_burned: int) -> dict:
    """
    Create a short motivational message using an LLMChain and then translate it to French using runnable2.
    Return a dict: {"motivation_en": <english message>, "motivation_fr": <french message>}
    Hints:
      - Create a PromptTemplate like "Write a short, upbeat motivational message for {user} who just burned {calories} doing {activity}."
      - Build an LLMChain and invoke it with the required variables. Pass the `callbacks` variable during invocation.
      - Use runnable2.invoke({"text": ...}) to get the translation.
    """
    raise NotImplementedError  # TODO: implement chain and runnable2 translation


# Example usage (uncomment and run after implementing the functions):
# activity_name = "running"
# cals_per_min = 15
# duration = 30
#
# # 1) Find user's name in memory
# user = get_user_name_from_memory()
# print(f"User found in memory: {user}")
#
# # 2) Calculate calories burned
# total_calories = calculate_calories_burned(cals_per_min, duration)
# print(f"Total calories burned: {total_calories}")
#
# # 3) Generate motivational message + French translation
# messages = generate_motivational_message(user, activity_name, total_calories)
# print("Motivation (EN):", messages["motivation_en"])

#
# Expected outcome:
# - A user name extracted (e.g., "Bob")
# - A correct calculation of total calories (e.g., 450)
# - A short motivational message in English

In [ ]:
from typing import Optional

# Exercise (Retail domain, ~10 min)
# Objective:
# Build a small "Retail Promotion Assistant" that:
#  1. Reads a returning customer's name from conversation memory.
#  2. Computes a discounted price using the existing tool `multiply` (work in cents to stay integer-safe).
#  3. Generates a one-line promotional message using the existing `llm` (via an LLMChain + PromptTemplate).
#  4. Produces a French translation of the promo using the existing runnable `runnable2`.
#
# Constraints / hints:
#  - Use `buffer_memory` (or `memory`) to find a recent Human message like "my name is bob".
#  - multiply.func(a: int, b: int) is available on the StructuredTool `multiply`. Work in cents:
#      discounted_cents = multiply.func(price_cents, 100 - discount_percent) // 100
#  - Build a PromptTemplate for the one-line promo and construct an LLMChain with the existing `llm`.
#  - Use `runnable2.invoke({"text": text})` to get the French translation (hint: runnable2 expects input variable 'text').
#  - Use `callbacks` (already defined) when invoking chains to demonstrate callback use.
#
# Tasks (implement the three functions below). Do not change global objects (llm, multiply, buffer_memory, runnable2, callbacks).


def compute_discounted_price(price: float, discount_percent: int) -> float:
    """
    Compute discounted price using the `multiply` tool in integer cents to avoid float rounding issues.
    Return price as dollars (float).
    Hint:
      price_cents = int(round(price * 100))
      product = multiply.func(price_cents, 100 - discount_percent)
      discounted_cents = product // 100
    """
    raise NotImplementedError  # TODO: implement using multiply.func


def get_customer_name_from_memory(mem=buffer_memory) -> Optional[str]:
    """
    Inspect mem.chat_memory.messages (list of HumanMessage/AIMessage objects).
    Common pattern to look for: a human message that contains 'my name is <name>' or 'I am <name>'.
    Return the found name (capitalize) or None if not found.
    Hints:
      - Each message object has .content (string).
      - A simple parsing approach is fine (split on 'my name is' or 'i am').
    """
    raise NotImplementedError  # TODO: implement simple parsing


def generate_promotion(customer_name: Optional[str], product: str, final_price: float, use_callbacks=True) -> dict:
    """
    Create a one-line promotional message using an LLMChain and then translate it to French using runnable2.
    Return a dict: {"promo": <english promo string>, "promo_fr": <french promo string>}
    Hints:
      - Use PromptTemplate.from_template("Write a concise one-line promo for {customer} about {product} priced at ${price}.")
      - Build LLMChain(llm=llm, prompt=your_prompt). Invoke with {"customer": ..., "product": ..., "price": ...}
      - Pass callbacks variable when invoking (callbacks exists).
      - runnable2.invoke({"text": promo_text}) -> translation
    """
    raise NotImplementedError  # TODO: implement chain + runnable2 translation


# Example usage (uncomment and run after implementing the functions):
# product_name = "Wireless Mouse"
# original_price = 49.99
# discount_pct = 20
#
# # 1) find customer
# cust = get_customer_name_from_memory()
# print("Customer found in memory:", cust)
#
# # 2) compute discounted price
# new_price = compute_discounted_price(original_price, discount_pct)
# print(f"Discounted price for {product_name}: ${new_price:.2f}")
#
# # 3) generate promo + French translation
# result = generate_promotion(cust, product_name, new_price)
# print("Promo (EN):", result["promo"])
# print("Promo (FR):", result["promo_fr"])
#
# Expected outcome:
# - A customer name extracted (e.g., "Bob" if present in buffer_memory)
# - A discounted price computed correctly
# - A one-line promotional message and its French translation
#
# Grading hints:
# - 2 pts: correctly extracts a name from memory (or returns None gracefully)
# - 3 pts: computes discounted price using multiply (correct cents math)
# - 5 pts: generates a one-line promo and a French translation using the provided LLM runnable
#
# Optional extension (stretch): Use runnable1 to also produce a short summary of the product and include it in the promo chain prompt.